# Eficácia do Pressing no Futebol de Elite — Análise de Gatilhos de pressão

## 1. Introdução

Este notebook busca apresentar uma análise da eficácia de determinados gatilhos de pressão no futebol de alto nível, complementando a análise geral de pressão, em especial após determinados tipos de passes e a consequência dos mesmos.

## 2. Configuração e Carregamento de Dados

In [1]:
import json
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


In [2]:
DATA_DIR = Path('StatsBomb_2/data')
if not DATA_DIR.exists():
    DATA_DIR = Path('C:\\Users\\DELL\\Downloads\\CDAF - bases de dados e codigos de apoio\\data\\archive\\data')

COMP_ID, SEASON_ID = 55, 43  # UEFA Euro 2020
FIELD_X, FIELD_Y = 120.0, 80.0

def load_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)

matches = load_json(DATA_DIR / 'matches' / str(COMP_ID) / (str(SEASON_ID) + '.json'))
match_ids = sorted(m['match_id'] for m in matches)

events_parts = []
frames = {}
for mid in match_ids:
    raw = load_json(DATA_DIR / 'events' / (str(mid) + '.json'))
    edf = pd.json_normalize(raw, sep='_')
    edf['match_id'] = mid
    events_parts.append(edf)
    ts_path = DATA_DIR / 'three-sixty' / (str(mid) + '.json')
    if ts_path.exists():
        for fr in load_json(ts_path):
            frames[fr['event_uuid']] = fr

events = pd.concat(events_parts, ignore_index=True)
def ts_to_seconds(ts):
    hh, mm, ss = ts.split(':')
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

events['t'] = events['timestamp'].apply(ts_to_seconds)
events['t_match'] = events['minute'] * 60 + events['second']
print(f'Eventos carregados: {len(events)} | Frames 360: {len(frames)}')

Eventos carregados: 192692 | Frames 360: 166892


## 3. Escolha e Especificação de Gatilhos

É notável a ausência na literatura de estudos aprofundados que definam o que e como caracterizam gatilhos de pressão. Artigos como o de **Harrison (2025)** oferecem exemplos visuais de _triggers_ que aumentam a frequência de eventos de pressão, mas não justificam a escolha dos gatilhos. Por isso, este estudo foca em estabelecer critérios básicos para definir gatilhos que possam ser utilizados para a análise de eficácia.

É importante observar que, como os dados Statsbomb 360 utilizam-se de eventos para definir um _freeze frame_ da partida, uma análise que considere fatores como erros de domínio de bola e tentativas de drible, que também são fatores que podem se tornar gatilhos, não é considerada viável. Por isso, este notebook se dedica a estabelecer uma análise sólida e estruturada de diferentes gatilhos de passes.

### 3.1. Definição de um _Trigger_

Um evento de pressão no Statsbomb ocorre no instante em que um jogador estiver a um raio de 5 jardas de um adversário que tenha a posse da bola (**Merckx et al., 2021**). Uma pressão aplicada de um jogador sobre outro pode ser facilmente observável por esse evento, mas, para esta análise, será considerada uma pressão acionada por gatilho apenas aquela que, nos 10 segundos seguintes, gerar no mínimo outro evento de pressão por outro jogador ou recuperação de bola pela equipe que pressiona, e não haver outro evento de pressão coletiva observado nos 10 segundos anteriores (para evitar pressões duplicadas).

Tendo esse evento de pressão coletiva definido, o gatilho será classificado a partir do passe que originou a pressão, de forma que consideramos apenas os eventos definidos de pressão coletiva que resultaram de um passe nos 6 segundos anteriores. Definimos como passes característicos de gerar _triggers_ como recuos, passes laterais onde não há progressão, passes próximos à linha lateral com bola rolando e passes especificamente destinados aos goleiros.

### 3.1.1. Recuos

São definidos como passes que têm como destino uma posição vertical inferior em relação à origem. para desconsiderar passes laterais comuns, são considerados apenas aqueles que regridem pelo menos 10 metros no campo.

### 3.1.2. Passes laterais sem progressão

Passes que têm pouco impacto no aspecto vertical do jogo. São analisados aqueles que percorrem ao menos 10 metros horizontalmente e menos de 10 metros verticalmente, tanto para frente quanto para trás.

### 3.1.3. Passes próximos à linha lateral

Passes cujo receptor está em até 5 metros de distância da linha lateral, excluindo cobranças de lateral.

### 3.1.4. Cobranças de lateral

Único tipo de bola parada analisada, já que gera pressão em grande parte das execuções.

### 3.1.5. Passes para o goleiro

Exemplo clássico de gatilho de pressão, precisa apenas de um jogador pressionando diferentemente dos outros, ocorre quando o jogador em posse no instante da pressão é o goleiro adversário.

### 3.1.6. Adversários sem companheiros de equipe próximos

Diferentemente dos outros, não depende dos eventos de passe. Ocorre quando não houver nenhum companheiro de equipe em um raio de 15m de distância do jogador com a bola.

In [3]:
#Separando os eventos de pressão
pressures = events[events['type_name'] == 'Pressure'].copy()
passes = events[events['type_name'] == 'Pass'].copy()
print(f'Eventos de pressão : {len(pressures)} \nEventos de passes: {len(passes)}')

Eventos de pressão : 15958 
Eventos de passes: 54819


In [ ]:
# Separando os eventos que caracterizam uma pressão coletiva

PRESSURE_WINDOW = 10

pressures = pressures.sort_values(['match_id', 'period', 't', 'index']).copy()
pressures['pressure_event_id'] = pressures['id']
events_sorted = events.sort_values(['match_id', 'period', 't', 'index']).copy()

possession_team_col = None
for candidate_col in ['possession_team_name', 'possession_team']:
    if candidate_col in events_sorted.columns:
        possession_team_col = candidate_col
        break

RECOVERY_EVENT_TYPES = {
    'Ball Recovery',
    'Interception',
    'Duel',
    'Dribbled Past',
    'Miscontrol',
    'Dispossessed',
    'Error',
    'Clearance',
    'Block'
}

def get_events_after_pressure(row, df=events_sorted, window=PRESSURE_WINDOW):
    """Retorna eventos ocorridos após a pressão dentro da janela definida."""
    window_events = df[
        (df['match_id'] == row['match_id']) &
        (df['period'] == row['period']) &
        (df['t'] > row['t']) &
        (df['t'] <= row['t'] + window)
    ].copy()

    if 'index' in window_events.columns and 'index' in row.index and not pd.isna(row['index']):
        window_events = window_events[window_events['index'] > row['index']]

    return window_events

def has_next_team_pressure(row, df=pressures, window=PRESSURE_WINDOW):
    """Verifica se há nova pressão da mesma equipe por outro jogador nos 10s seguintes."""
    mask = (
        (df['match_id'] == row['match_id']) &
        (df['period'] == row['period']) &
        (df['team_name'] == row['team_name']) &
        (df['t'] > row['t']) &
        (df['t'] <= row['t'] + window) &
        (df['player_id'] != row['player_id'])
    )

    if 'index' in df.columns and 'index' in row.index and not pd.isna(row['index']):
        mask = mask & (df['index'] > row['index'])

    return bool(mask.any())

def has_recovery_within_window(row, window=PRESSURE_WINDOW):
    """Verifica se o time que pressiona recupera a posse nos 10s seguintes."""
    window_events = get_events_after_pressure(row, window=window)

    if window_events.empty:
        return False

    pressure_team = row['team_name']

    # Critério preferencial: a posse passa a ser da equipe que pressionou.
    if possession_team_col is not None:
        return bool(window_events[possession_team_col].eq(pressure_team).any())

    # Fallback: eventos explícitos de recuperação/interceptação/duelo da equipe que pressionou.
    own_recovery = window_events[
        (window_events['team_name'] == pressure_team) &
        (window_events['type_name'].isin(RECOVERY_EVENT_TYPES))
    ]
    if not own_recovery.empty:
        return True

    # Fallback mínimo: algum evento posterior da equipe que pressionou.
    return bool((window_events['team_name'] == pressure_team).any())

pressures['has_next_team_pressure_10s'] = pressures.apply(has_next_team_pressure, axis=1)
pressures['has_recovery_10s'] = pressures.apply(has_recovery_within_window, axis=1)
pressures['has_next_team_pressure_or_recovery_10s'] = (
    pressures['has_next_team_pressure_10s'] | pressures['has_recovery_10s']
)

team_pressures = pressures[pressures['has_next_team_pressure_or_recovery_10s']].copy()

print(f'Eventos com outra pressão por outro jogador em até 10s: {int(pressures["has_next_team_pressure_10s"].sum())}')
print(f'Eventos com recuperação em até 10s: {int(pressures["has_recovery_10s"].sum())}')
print(f'Eventos de pressão coletiva/efetiva: {len(team_pressures)}')

Eventos com outra pressão por outro jogador em até 10s: 6149
Eventos com recuperação em até 10s: 4948
Eventos de pressão coletiva/efetiva: 9574


In [ ]:
# Filtrando os eventos de início de pressão coletiva
# Mantém apenas o primeiro evento de uma sequência, evitando duplicidade.

def has_previous_collective_pressure(row, df=team_pressures, window=PRESSURE_WINDOW):
    mask = (
        (df['match_id'] == row['match_id']) &
        (df['period'] == row['period']) &
        (df['team_name'] == row['team_name']) &
        (df['t'] < row['t']) &
        (df['t'] >= row['t'] - window)
    )
    return bool(mask.any())

team_pressures['has_previous_collective_pressure_10s'] = team_pressures.apply(
    has_previous_collective_pressure, axis=1
)

team_pressures_base = team_pressures[
    ~team_pressures['has_previous_collective_pressure_10s']
].copy()

print(f'Eventos de início pressão coletiva: {len(team_pressures_base)}')

Eventos de início pressão coletiva: 5687


In [ ]:
# Selecionando apenas os eventos que são consequência de passes
# Para cada início de pressão coletiva, seleciona-se o passe adversário mais recente
# ocorrido até 6s antes do início da pressão.

PASS_TRIGGER_WINDOW = 6

passes = passes.sort_values(['match_id', 'period', 't', 'index']).copy()
completed_passes = passes[
    passes.get('pass_outcome_name', pd.Series(index=passes.index)).isna()
].copy()

def find_previous_opponent_pass(row, df=completed_passes, window=PASS_TRIGGER_WINDOW):
    cand = df[
        (df['match_id'] == row['match_id']) &
        (df['period'] == row['period']) &
        (df['team_name'] != row['team_name']) &
        (df['t'] >= row['t'] - window) &
        (df['t'] <= row['t'])
    ]
    if cand.empty:
        return pd.Series(dtype='object')
    return cand.sort_values(['t', 'index']).iloc[-1]

trigger_rows = []
for _, pressure_row in team_pressures_base.iterrows():
    pass_row = find_previous_opponent_pass(pressure_row)
    if pass_row.empty:
        continue

    combined = pressure_row.add_prefix('pressure_').to_dict()
    combined.update(pass_row.add_prefix('pass_').to_dict())
    combined['seconds_from_pass_to_pressure'] = pressure_row['t'] - pass_row['t']
    trigger_rows.append(combined)

passing_triggers = pd.DataFrame(trigger_rows)

print(f'Eventos de gatilhos de passe: {len(passing_triggers)}')

Eventos de gatilhos de passe: 3551


In [ ]:
# Recuos
# Passes cuja coordenada x final é pelo menos 10m menor que a inicial.

MIN_BACKWARD_METERS = 10

def get_xy(value):
    if isinstance(value, (list, tuple, np.ndarray)) and len(value) >= 2:
        return float(value[0]), float(value[1])
    return np.nan, np.nan

if not passing_triggers.empty:
    passing_triggers[['pass_start_x', 'pass_start_y']] = passing_triggers['pass_location'].apply(
        lambda v: pd.Series(get_xy(v))
    )
    passing_triggers[['pass_end_x', 'pass_end_y']] = passing_triggers['pass_pass_end_location'].apply(
        lambda v: pd.Series(get_xy(v))
    )
    passing_triggers['pass_dx'] = passing_triggers['pass_end_x'] - passing_triggers['pass_start_x']
    passing_triggers['pass_dy'] = passing_triggers['pass_end_y'] - passing_triggers['pass_start_y']
else:
    for col in ['pass_start_x', 'pass_start_y', 'pass_end_x', 'pass_end_y', 'pass_dx', 'pass_dy']:
        passing_triggers[col] = pd.Series(dtype='float')

backpasses = passing_triggers[
    passing_triggers['pass_dx'] <= -MIN_BACKWARD_METERS
].copy()

print(f'Eventos de recuos: {len(backpasses)}')

Eventos de recuos: 251


In [ ]:
# Passes laterais
# Passes com deslocamento horizontal relevante e baixa progressão/regressão:
# - pelo menos 10m no eixo y;
# - menos de 10m, em módulo, no eixo x.

MIN_HORIZONTAL_METERS = 10
MAX_VERTICAL_CHANGE_METERS = 10

lateral_passes = passing_triggers[
    (passing_triggers['pass_dy'].abs() >= MIN_HORIZONTAL_METERS) &
    (passing_triggers['pass_dx'].abs() < MAX_VERTICAL_CHANGE_METERS)
].copy()

print(f'Eventos de passes laterais: {len(lateral_passes)}')

Eventos de passes laterais: 857


In [ ]:
# Passes próximos à linha lateral
# Exclui cobranças de lateral para manter apenas bola rolando.

# Receptor em até 5m de uma das linhas laterais.
SIDELINE_DISTANCE = 5
FIELD_Y = 80.0

pass_type_col = 'pass_pass_type_name'
is_throw_in_pass = (
    passing_triggers[pass_type_col].eq('Throw-in')
    if pass_type_col in passing_triggers.columns
    else pd.Series(False, index=passing_triggers.index)
)

sideline_passes = passing_triggers[
    (
        (passing_triggers['pass_end_y'] <= SIDELINE_DISTANCE) |
        (passing_triggers['pass_end_y'] >= FIELD_Y - SIDELINE_DISTANCE)
    ) &
    (~is_throw_in_pass)
].copy()

print(f'Eventos de passes próximos à linha lateral: {len(sideline_passes)}')

Eventos de passes próximos à linha lateral: 466


In [ ]:
# Cobranças de lateral
# passes cujo tipo StatsBomb é Throw-in.

pass_type_col = 'pass_pass_type_name'

if pass_type_col in passing_triggers.columns:
    throw_ins = passing_triggers[
        passing_triggers[pass_type_col].eq('Throw-in')
    ].copy()
else:
    throw_ins = passing_triggers.iloc[0:0].copy()
    print("Coluna pass_pass_type_name não encontrada; não foi possível filtrar cobranças de lateral.")

print(f'Eventos de cobranças de lateral: {len(throw_ins)}')

Eventos de cobranças de lateral: 353


In [ ]:
# Passes para o goleiro
# localiza-se o adversário mais próximo do pressionador no instante da pressão;
# se esse adversário for goleiro, é classificado como gatilho de pressão no goleiro.

def nearest_opponent_to_pressure(pressure_event_id, pressure_team_is_teammate=True):
    frame = frames.get(pressure_event_id)
    if not frame or 'freeze_frame' not in frame:
        return pd.Series({
            'ball_carrier_name': np.nan,
            'ball_carrier_is_keeper': np.nan,
            'ball_carrier_x': np.nan,
            'ball_carrier_y': np.nan,
            'ball_carrier_teammates_15m': np.nan
        })

    ff = frame['freeze_frame']

    actor = next((p for p in ff if p.get('actor')), None)
    if actor is None:
        return pd.Series({
            'ball_carrier_name': np.nan,
            'ball_carrier_is_keeper': np.nan,
            'ball_carrier_x': np.nan,
            'ball_carrier_y': np.nan,
            'ball_carrier_teammates_15m': np.nan
        })

    ax, ay = actor.get('location', [np.nan, np.nan])[:2]

    # No freeze frame, teammate=True costuma ser relativo à equipe do evento.
    # Como o evento é Pressure, teammate=False representa adversários pressionados.
    opponents = [p for p in ff if not p.get('teammate', False)]
    if not opponents:
        return pd.Series({
            'ball_carrier_name': np.nan,
            'ball_carrier_is_keeper': np.nan,
            'ball_carrier_x': np.nan,
            'ball_carrier_y': np.nan,
            'ball_carrier_teammates_15m': np.nan
        })

    def dist_to_actor(p):
        x, y = p.get('location', [np.nan, np.nan])[:2]
        return np.hypot(x - ax, y - ay)

    ball_carrier = min(opponents, key=dist_to_actor)
    bx, by = ball_carrier.get('location', [np.nan, np.nan])[:2]
    position_name = ball_carrier.get('position', {}).get('name', '')
    is_keeper = bool(ball_carrier.get('keeper', False)) or position_name == 'Goalkeeper'

    teammates_15m = 0
    for p in opponents:
        if p is ball_carrier:
            continue
        x, y = p.get('location', [np.nan, np.nan])[:2]
        if np.hypot(x - bx, y - by) <= 15:
            teammates_15m += 1

    return pd.Series({
        'ball_carrier_name': ball_carrier.get('player', {}).get('name', np.nan),
        'ball_carrier_is_keeper': is_keeper,
        'ball_carrier_x': bx,
        'ball_carrier_y': by,
        'ball_carrier_teammates_15m': teammates_15m
    })

if not passing_triggers.empty:
    pressure_ids = passing_triggers['pressure_id']
    ball_carrier_info = pressure_ids.apply(nearest_opponent_to_pressure)
    passing_triggers = pd.concat(
        [passing_triggers.reset_index(drop=True), ball_carrier_info.reset_index(drop=True)],
        axis=1
    )

gk_passes = passing_triggers[
    passing_triggers['ball_carrier_is_keeper'].fillna(False)
].copy()

print(f'Eventos de passes para o goleiro: {len(gk_passes)}')

Eventos de passes para o goleiro: 56


In [ ]:
# Adversários sem companheiros próximos
# É calculado sobre os inícios de pressão coletiva, não apenas sobre passing_triggers.


# Jogador pressionado sem companheiro de equipe em raio de 15m.
ISOLATION_RADIUS = 15

if not team_pressures_base.empty:
    isolated_info = team_pressures_base['id'].apply(nearest_opponent_to_pressure)
    team_pressures_base_with_context = pd.concat(
        [team_pressures_base.reset_index(drop=True), isolated_info.reset_index(drop=True)],
        axis=1
    )
else:
    team_pressures_base_with_context = team_pressures_base.copy()

open_area_passes = team_pressures_base_with_context[
    team_pressures_base_with_context['ball_carrier_teammates_15m'].eq(0)
].copy()

print(f'Eventos de passes para jogadores isolados: {len(open_area_passes)}')

Eventos de passes para jogadores isolados: 600


## 3.1. Taxa de recuperação após os gatilhos

Nesta etapa, cada início de pressão coletiva/efetiva e cada gatilho filtrado recebe a variável `recovered`, definida como `True` quando o time que pressiona recupera a posse em até 5 segundos após o início da pressão.

A recuperação ocorre quando, dentro da janela temporal, aparece pelo menos um evento cuja equipe em posse (`possession_team_name`) passa a ser a equipe que iniciou a pressão. Caso a coluna de posse não esteja disponível, o código usa como alternativa eventos explícitos de recuperação, interceptação, duelo vencido ou erro/controle perdido do adversário seguido de evento da equipe que pressionou.

In [20]:
# Taxa de sucesso na recuperação de bola após pressão coletiva e gatilhos
# Definição de sucesso/recovered:
# recovered = True se a equipe que pressiona recuperar a posse em até 3s após o início da pressão.
# A janela parte do evento de início da pressão coletiva, não do passe anterior.

RECOVERY_WINDOW = 5

# Coluna mais adequada para identificar a equipe em posse.
# Nos dados StatsBomb, normalmente ela existe como possession_team_name.
possession_team_col = None
for candidate_col in ['possession_team_name', 'possession_team']:
    if candidate_col in events.columns:
        possession_team_col = candidate_col
        break

RECOVERY_EVENT_TYPES = {
    'Ball Recovery',
    'Interception',
    'Duel',
    'Dribbled Past',
    'Miscontrol',
    'Dispossessed',
    'Error',
    'Clearance',
    'Block'
}

def _row_value(row, unprefixed_col, prefixed_col=None):
    """Lê coluna em dataframes de pressão pura ou de gatilhos com prefixo pressure_."""
    prefixed_col = prefixed_col or f'pressure_{unprefixed_col}'
    if prefixed_col in row.index:
        return row[prefixed_col]
    if unprefixed_col in row.index:
        return row[unprefixed_col]
    return np.nan

def recovered_after_pressure(row, window=RECOVERY_WINDOW):
    """Retorna True se o time que pressiona recuperar a posse em até window segundos."""
    match_id = _row_value(row, 'match_id')
    period = _row_value(row, 'period')
    pressure_t = _row_value(row, 't')
    pressure_team = _row_value(row, 'team_name')
    pressure_index = _row_value(row, 'index')

    if pd.isna(match_id) or pd.isna(period) or pd.isna(pressure_t) or pd.isna(pressure_team):
        return False

    window_events = events[
        (events['match_id'] == match_id) &
        (events['period'] == period) &
        (events['t'] > pressure_t) &
        (events['t'] <= pressure_t + window)
    ].copy()

    if 'index' in window_events.columns and not pd.isna(pressure_index):
        window_events = window_events[window_events['index'] > pressure_index]

    if window_events.empty:
        return False

    # Troca de posse
    if possession_team_col is not None:
        return bool(window_events[possession_team_col].eq(pressure_team).any())

    # Fallback se a coluna de posse não existir
    # conta eventos explícitos de recuperação/interceptação/duelo da equipe que pressionou.
    own_recovery = window_events[
        (window_events['team_name'] == pressure_team) &
        (window_events['type_name'].isin(RECOVERY_EVENT_TYPES))
    ]
    if not own_recovery.empty:
        return True

    # Alternativa- primeiro evento relevante da equipe que pressionou após uma ação do adversário.
    first_own_event = window_events[window_events['team_name'] == pressure_team]
    return bool(not first_own_event.empty)

def add_recovered_column(df, label):
    """Adiciona recovered e imprime resumo básico."""
    if df is None or df.empty:
        out = df.copy() if df is not None else pd.DataFrame()
        out['recovered'] = pd.Series(dtype='bool')
        print(f'{label}: 0 eventos | recovered = n/a')
        return out

    out = df.copy()
    out['recovered'] = out.apply(recovered_after_pressure, axis=1)
    rate = out['recovered'].mean()
    print(f'{label}: {len(out)} eventos | recuperações = {int(out["recovered"].sum())} | taxa = {rate:.2%}')
    return out

# Todos os inícios de pressão coletiva
team_pressures_base = add_recovered_column(team_pressures_base, 'Todos os eventos de pressão coletiva')

# Versão com contexto espacial, caso já tenha sido criada na célula de isolamento
if 'team_pressures_base_with_context' in globals():
    team_pressures_base_with_context = add_recovered_column(
        team_pressures_base_with_context,
        'Todos os eventos de pressão coletiva com contexto 360'
    )

# Gatilhos derivados de passes
passing_triggers = add_recovered_column(passing_triggers, 'Todos os gatilhos após passe')
backpasses = add_recovered_column(backpasses, 'Recuos')
lateral_passes = add_recovered_column(lateral_passes, 'Passes laterais')
sideline_passes = add_recovered_column(sideline_passes, 'Passes próximos à linha lateral')
throw_ins = add_recovered_column(throw_ins, 'Cobranças de lateral')
gk_passes = add_recovered_column(gk_passes, 'Passes para o goleiro')
open_area_passes = add_recovered_column(open_area_passes, 'Jogadores isolados')

# Tabela-resumo da Etapa 3
trigger_summary = pd.DataFrame([
    #{'trigger': 'pressao_coletiva_total', 'n': len(team_pressures_base), 'recovered_n': int(team_pressures_base['recovered'].sum()), 'recovered_rate': team_pressures_base['recovered'].mean()},
    {'trigger': 'gatilhos_apos_passe', 'n': len(passing_triggers), 'recovered_n': int(passing_triggers['recovered'].sum()), 'recovered_rate': passing_triggers['recovered'].mean()},
    {'trigger': 'recuos', 'n': len(backpasses), 'recovered_n': int(backpasses['recovered'].sum()), 'recovered_rate': backpasses['recovered'].mean()},
    {'trigger': 'passes_laterais', 'n': len(lateral_passes), 'recovered_n': int(lateral_passes['recovered'].sum()), 'recovered_rate': lateral_passes['recovered'].mean()},
    {'trigger': 'passes_linha_lateral', 'n': len(sideline_passes), 'recovered_n': int(sideline_passes['recovered'].sum()), 'recovered_rate': sideline_passes['recovered'].mean()},
    {'trigger': 'cobrancas_lateral', 'n': len(throw_ins), 'recovered_n': int(throw_ins['recovered'].sum()), 'recovered_rate': throw_ins['recovered'].mean()},
    {'trigger': 'passes_para_goleiro', 'n': len(gk_passes), 'recovered_n': int(gk_passes['recovered'].sum()), 'recovered_rate': gk_passes['recovered'].mean()},
    {'trigger': 'jogadores_isolados', 'n': len(open_area_passes), 'recovered_n': int(open_area_passes['recovered'].sum()), 'recovered_rate': open_area_passes['recovered'].mean()},
])

trigger_summary['recovered_rate_pct'] = (trigger_summary['recovered_rate'] * 100).round(2)
trigger_summary = trigger_summary.sort_values('recovered_rate', ascending=False)

trigger_summary


Todos os eventos de pressão coletiva: 5687 eventos | recuperações = 2487 | taxa = 43.73%
Todos os eventos de pressão coletiva com contexto 360: 5687 eventos | recuperações = 2487 | taxa = 43.73%
Todos os gatilhos após passe: 3551 eventos | recuperações = 1068 | taxa = 30.08%
Recuos: 251 eventos | recuperações = 51 | taxa = 20.32%
Passes laterais: 857 eventos | recuperações = 233 | taxa = 27.19%
Passes próximos à linha lateral: 466 eventos | recuperações = 134 | taxa = 28.76%
Cobranças de lateral: 353 eventos | recuperações = 99 | taxa = 28.05%
Passes para o goleiro: 56 eventos | recuperações = 9 | taxa = 16.07%
Jogadores isolados: 600 eventos | recuperações = 226 | taxa = 37.67%


,trigger,n,recovered_n,recovered_rate,recovered_rate_pct
6,jogadores_isolados,600,226,0.376667,37.67
0,gatilhos_apos_passe,3551,1068,0.300760,30.08
3,passes_linha_lateral,466,134,0.287554,28.76
4,cobrancas_lateral,353,99,0.280453,28.05
2,passes_laterais,857,233,0.271879,27.19
1,recuos,251,51,0.203187,20.32
5,passes_para_goleiro,56,9,0.160714,16.07


# 4. Análise avançada dos triggers